# The Anatomy of AI · Lecture 18 · Painting with Concepts — the notebook

Lecture 17's prism found clean units inside a network that *sees*. Here you put prisms to work in a network that
**paints** and in networks that **see**, with the published prisms the lecture used (nobody needs to retrain them):

1. open SDXL-Turbo, an image generator, while it paints a sloth, and look inside one of its blocks;
2. **paint** a prism unit onto chosen patches of the picture, and measure how local the stroke is at three blocks;
3. open a dragon's mouth with Goodfire's "animal opening its mouth" unit;
4. take the stripes out of zebras, and put them into horses, inside CLIP and DINOv2.

Every key number is checked against the lecture's own. It needs a GPU (*Runtime → Change runtime type → T4 GPU*) and
takes about 15 minutes on free Colab, most of it downloads (SDXL-Turbo is about 7 GB).

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "pillow==11.1.0", "pyarrow==18.1.0", "requests==2.32.3", "diffusers==0.31.0", "transformers==4.46.3", "accelerate==1.1.1", "huggingface_hub==0.26.2", "safetensors==0.4.5", "open_clip_torch==2.26.1")

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")


def fetch(url, path, what):
    """Download a file once (quietly), and say in one line what came down."""
    import requests
    if not os.path.exists(path):
        t = time.time()
        with requests.get(url, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded {what} ({os.path.getsize(path) / 1e6:.1f} MB) in {time.time() - t:.0f} s")
    return path

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"          # one line per download, not a percentage bar
os.environ["TQDM_DISABLE"] = "1"                          # no progress bars (or their widgets) from any library
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"                 # no library notices on the page
os.environ["DIFFUSERS_VERBOSITY"] = "error"
import contextlib, gc, logging
for name in ("diffusers", "transformers", "huggingface_hub", "open_clip"):
    logging.getLogger(name).setLevel(logging.ERROR)
assert DEV == "cuda", "this notebook needs a GPU: Runtime → Change runtime type → T4 GPU"

## Part 1 · A network that paints

### 1. A modern painter (Plate II)
**SDXL-Turbo** turns a sentence into a picture. It starts from a block of random numbers (64 × 64 × 4), runs a
**U-Net** over it *once* (ordinary diffusion runs it many times, removing a little noise each time), and a decoder blows
the cleaned 64 × 64 × 4 *latent* up to 512 × 512 pixels. We use the lecture's sentence and seed.

In [ ]:
with contextlib.redirect_stderr(io.StringIO()):            # a fresh machine prints a one-time cache notice here
    from diffusers import AutoPipelineForText2Image
    import diffusers, transformers
diffusers.utils.logging.set_verbosity_error(); transformers.logging.set_verbosity_error()
diffusers.utils.logging.disable_progress_bar(); transformers.utils.logging.disable_progress_bar()
t = time.time()
with contextlib.redirect_stderr(io.StringIO()):
    pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16").to(DEV)
pipe.set_progress_bar_config(disable=True)
print(f"SDXL-Turbo (stabilityai/sdxl-turbo, half precision) ready in {time.time() - t:.0f} s")
unet = pipe.unet
SLOTH = "A cinematic shot of a professor sloth wearing a tuxedo at a BBQ party."


def generate(prompt, seed=42, hooks=()):
    # one step, no guidance, 512 x 512: how the lecture made every picture. hooks: [(module, fn)] to look or edit inside
    hs = [m.register_forward_hook(f) for m, f in hooks]
    try:
        g = torch.Generator("cpu").manual_seed(seed)
        return pipe(prompt, num_inference_steps=1, guidance_scale=0.0, generator=g, height=512, width=512).images[0]
    finally:
        for h in hs:
            h.remove()

In [ ]:
seen = {}
h1 = unet.register_forward_pre_hook(lambda m, a: seen.__setitem__("noise", a[0].float().cpu()))
sloth = generate(SLOTH, 42)
h1.remove()
noise = seen["noise"]
print("the U-Net's input:", list(noise.shape), f"· spread {noise.std():.3f}")
check("the starting noise's shape", list(noise.shape), [1, 4, 64, 64])
check("the starting noise's spread", noise.std().item(), 0.999, tol=0.005)
fig, axs = plt.subplots(1, 5, figsize=(11, 2.4))
for c in range(4):
    axs[c].imshow(noise[0, c], cmap="gray"); axs[c].axis("off"); axs[c].set_title(f"noise, channel {c}", fontsize=8)
axs[4].imshow(sloth); axs[4].axis("off"); axs[4].set_title("out comes: 512 x 512", fontsize=8)
plt.show()

### 2. What a block adds (Plate III)
Deep in the U-Net, at its narrowest blocks, the picture is a grid of 16 × 16 **patches**, 1,280 numbers each. Each
block *adds* something to what comes in; that addition is its **residual update**. Surkov et al. (2024) trained prisms on
what four of these blocks add: 5,120 units, of which only the 10 strongest may be on at a patch (*TopK*). We load their
published prism for block **up.0.1** and rebuild its update on the sloth.

In [ ]:
from huggingface_hub import snapshot_download

BLOCKS = {"down.2.1": lambda: unet.down_blocks[2].attentions[1], "up.0.0": lambda: unet.up_blocks[0].attentions[0],
          "up.0.1": lambda: unet.up_blocks[0].attentions[1]}
FULL = {"down.2.1": "unet.down_blocks.2.attentions.1", "up.0.0": "unet.up_blocks.0.attentions.0",
        "up.0.1": "unet.up_blocks.0.attentions.1"}


class Prism:
    """Surkov et al.'s prism: f = ReLU(top 10 of W_enc (x - b_pre) + b_lat); x' = W_dec f + b_pre."""
    def __init__(self, block, k=10, n=5120):
        folder = f"{FULL[block]}_k{k}_hidden{n}_auxk256_bs4096_lr0.0001"
        root = snapshot_download("surokpro2/sdxl-saes", allow_patterns=[f"{folder}/state_dict.pth", f"{folder}/mean.pt"])
        sd = torch.load(f"{root}/{folder}/state_dict.pth", map_location="cpu", weights_only=False)
        sd = sd.get("state_dict", sd)
        self.We, self.Wd = sd["encoder.weight"].to(DEV).float(), sd["decoder.weight"].to(DEV).float()   # 5120x1280, 1280x5120
        self.bp, self.bl, self.k = sd["pre_bias"].to(DEV).float(), sd["latent_bias"].to(DEV).float(), k
        self.mu = torch.load(f"{root}/{folder}/mean.pt", map_location="cpu", weights_only=False).float()  # each unit's usual size

    def encode(self, x):
        pre = (x - self.bp) @ self.We.T + self.bl
        v, i = pre.topk(self.k, dim=-1)
        return torch.zeros_like(pre).scatter(-1, i, torch.relu(v))

    def decode(self, f):
        return f @ self.Wd.T + self.bp


def capture(store, key):
    def f(mod, inp, outp):
        o = outp[0] if isinstance(outp, tuple) else outp.sample
        store[key] = (inp[0].detach().float(), o.detach().float())
    return f


t = time.time()
prisms = {b: Prism(b) for b in ("up.0.1", "down.2.1")}
print(f"two of Surkov et al.'s prisms (surokpro2/sdxl-saes) ready in {time.time() - t:.0f} s")
store = {}
generate(SLOTH, 42, hooks=[(BLOCKS[b](), capture(store, b)) for b in prisms])
for b, pr in prisms.items():
    x_in, x_out = store[b]
    upd = (x_out - x_in)[0].permute(1, 2, 0).reshape(-1, 1280)       # 256 patches x 1,280 numbers
    f = pr.encode(upd)
    lost = float(((upd - pr.decode(f)) ** 2).sum() / ((upd - upd.mean(0)) ** 2).sum())
    print(f"{b}: block output {list(x_out.shape)}; {(f > 0).sum(1).float().mean():.0f} units on per patch; "
          f"the prism misses {100 * lost:.1f}% of what the block adds")
    check(f"{b}: block shape", list(x_out.shape), [1, 1280, 16, 16])
    check(f"{b}: the prism misses (variance lost)", lost, {"up.0.1": 0.3753, "down.2.1": 0.5707}[b], tol=0.02)
x_in, x_out = store["up.0.1"]
patch = (x_out - x_in)[0, :, 8, 3]
print(f"up.0.1, patch (8, 3): the block adds a list of length {patch.norm():.0f}")
check("up.0.1, patch (8, 3): length of what the block adds", patch.norm().item(), 179.7, tol=3)

### 3. One unit, one arrow, one brush (Plate IV)
To paint, pick a unit and a brush. At each brushed patch, add **strength × μ × the unit's arrow** (its decoder column) to
what the block sends on. μ is the unit's usual size when it is on. That is Lecture 14's move, adding an arrow, done only
where you brush. Unit **4977** of block up.0.1 is called "tiger stripes"; the brush is the left half.

To see how local a stroke is, compare the painting with the plain sloth, pixel by pixel: what share of all the change
lands inside the brush? (For a half-picture brush, chance is 50%.)

In [ ]:
REGIONS = {"full": (0, 16, 0, 16), "left": (0, 16, 0, 8), "right": (0, 16, 8, 16),
           "top": (0, 8, 0, 16), "bottom": (8, 16, 0, 16), "centre": (4, 12, 4, 12)}


def brush(name):
    r0, r1, c0, c1 = REGIONS[name]
    m = torch.zeros(16, 16, device=DEV); m[r0:r1, c0:c1] = 1
    return m


def paint_hook(arrow, amount):
    # amount: 16 x 16 (strength x mu x brush). Adds amount[i, j] x arrow at every patch (i, j) of the block's output.
    def f(mod, inp, outp):
        o = outp[0] if isinstance(outp, tuple) else outp.sample
        o2 = o + (amount[None, None] * arrow[None, :, None, None]).to(o.dtype)
        return (o2,) + tuple(outp[1:]) if isinstance(outp, tuple) else type(outp)(sample=o2)
    return f


def paint(block, unit, region, strength, prompt=SLOTH, seed=42):
    pr = prisms[block]
    amount = brush(region) * pr.mu[unit].item() * strength
    return generate(prompt, seed, hooks=[(BLOCKS[block](), paint_hook(pr.Wd[:, unit], amount))])


def locality(base, painted, region):
    # mean absolute change per pixel (0-255), inside vs outside the brush, and the share of all change inside
    d = np.abs(np.asarray(painted, np.float32) - np.asarray(base, np.float32)).mean(2)
    r0, r1, c0, c1 = REGIONS[region]
    m = np.zeros((512, 512), bool); m[r0 * 32:r1 * 32, c0 * 32:c1 * 32] = True
    return float(d[m].sum() / d.sum()), float(d[m].mean()), float(d[~m].mean()) if (~m).any() else float("nan")


pr = prisms["up.0.1"]
print(f"unit 4977: mu = {pr.mu[4977]:.2f}, arrow length {pr.Wd[:, 4977].norm():.2f}; strength 10 pushes each brushed "
      f"patch by {10 * pr.mu[4977] * pr.Wd[:, 4977].norm():.0f}, against the {patch.norm():.0f} the block itself added")
check("unit 4977's mu", pr.mu[4977].item(), 32.53, tol=0.01)
check("unit 4977's arrow length", pr.Wd[:, 4977].norm().item(), 1.0, tol=0.001)
stripes = paint("up.0.1", 4977, "left", 10)
share, inside, outside = locality(sloth, stripes, "left")
print(f"stripes on the left half: {100 * share:.0f}% of the change inside; per pixel {inside:.1f} inside, {outside:.1f} outside")
check("stripes on the left half: share of the change inside", share, 0.754, tol=0.05)
diff = np.abs(np.asarray(stripes, np.float32) - np.asarray(sloth, np.float32)).mean(2)
fig, axs = plt.subplots(1, 3, figsize=(10, 3.4))
for a, im, ttl in zip(axs, (sloth, stripes, np.clip((diff - 12) / 60, 0, 1)), ("before", "unit 4977, left half, strength 10", "what changed (dark = more)")):
    a.imshow(im, cmap="Greys" if ttl.startswith("what") else None); a.axis("off"); a.set_title(ttl, fontsize=9)
axs[2].axvline(255.5, color=INK, ls="--", lw=1)
plt.show()

### 4. Where you paint matters (Plate VI)
The same four half-picture brushes (left, right, top, bottom) at strength 10, with one unit at each of three blocks:
"tiger stripes" at up.0.1 (late), "beard" at up.0.0, "cartoon" at down.2.1 (early). How much of the change stays
inside the brush?

In [ ]:
prisms["up.0.0"] = Prism("up.0.0")
EX = {"up.0.1": 4977, "up.0.0": 3742, "down.2.1": 4998}
LECTURE_SHARES = {'up.0.1': [0.754, 0.772, 0.773, 0.81], 'up.0.0': [0.653, 0.65, 0.641, 0.72], 'down.2.1': [0.554, 0.591, 0.607, 0.712]}
LECTURE_MEDIAN = {'up.0.1': 0.772, 'up.0.0': 0.651, 'down.2.1': 0.599}
med = {}
fig, axs = plt.subplots(3, 4, figsize=(9, 7))
for row, (b, unit) in enumerate(EX.items()):
    shares = []
    for col, region in enumerate(("left", "right", "top", "bottom")):
        im = paint(b, unit, region, 10)
        s, _, _ = locality(sloth, im, region)
        shares.append(s)
        axs[row, col].imshow(im); axs[row, col].axis("off"); axs[row, col].set_title(f"{b} · {region}: {100 * s:.0f}%", fontsize=8)
        check(f"{b}, unit {unit}, {region}: share inside", s, LECTURE_SHARES[b][col], tol=0.06)
    med[b] = float(np.median(shares))
plt.tight_layout(); plt.show()
for b in EX:
    print(f"{b}: median {100 * med[b]:.0f}% of the change inside the brush (the lecture, these four: {100 * LECTURE_MEDIAN[b]:.0f}%)")
    check(f"{b}: median share inside", med[b], LECTURE_MEDIAN[b], tol=0.04)
check("late blocks keep the stroke in place better than early ones", med["up.0.1"] > med["up.0.0"] > med["down.2.1"], True)

### 5. An open mouth on a dragon (Plate VIII)
Goodfire's *Paint with Ember* (2025) published its own prism for block down.2.1: 10,240 units, reading the block's
output. Its unit **7446** is called "animal opening its mouth". First check the name: the unit's strongest patch on 24
pictures of animals with their mouths open, and 24 with them closed (six prompts, four seeds each). Then paint it onto a
dragon, at twice its open-mouth average, in the upper right of the picture, where the head is.

In [ ]:
from huggingface_hub import hf_hub_download
with contextlib.redirect_stderr(io.StringIO()):
    sd = torch.load(hf_hub_download("Goodfire/SDXL-Turbo-SAE-ldown.attns.2.1", "SDXL-Turbo-SAE-ldown_blocks.2.attentions.1.pt"),
                    map_location="cpu", weights_only=False)
gWe, gbe = sd["encoder.weight"].to(DEV).float(), sd["encoder.bias"].to(DEV).float()
gWd = sd["decoder.weight"].to(DEV).float()
print(f"Goodfire's prism: {gWe.shape[0]:,} units on block down.2.1's output")
OPEN = ["a lion roaring with its mouth wide open", "a dog barking with its mouth open", "a crocodile with its jaws wide open",
        "a cat yawning with its mouth wide open", "a wolf howling, mouth open", "a shark with its mouth open"]
CLOSED = ["a lion resting with its mouth closed", "a dog sitting calmly with its mouth closed", "a crocodile with its jaws shut",
          "a cat sleeping, mouth closed", "a wolf standing still, mouth closed", "a shark swimming, mouth closed"]


def unit7446(prompt, seed):
    st_ = {}
    generate(prompt, seed, hooks=[(BLOCKS["down.2.1"](), capture(st_, "b"))])
    a = torch.relu(st_["b"][1].permute(0, 2, 3, 1).reshape(-1, 1280) @ gWe.T + gbe)      # 256 patches x 10,240 units
    top10 = a.topk(10, dim=1).values[:, -1]
    return float(a[:, 7446].max()), bool((a[:, 7446] >= top10).any())


t = time.time()
res = {g: [unit7446(p, s) for p in ps for s in range(4)] for g, ps in (("open", OPEN), ("closed", CLOSED))}
for g in res:
    print(f"mouths {g}: strongest patch {np.mean([v for v, _ in res[g]]):.1f} on average; in the top 10 units "
          f"somewhere in {100 * np.mean([t_ for _, t_ in res[g]]):.0f}% of pictures")
print(f"({time.time() - t:.0f} s for 48 pictures)")
check("open mouths: the unit's strongest patch, average", float(np.mean([v for v, _ in res["open"]])), 107.0, tol=4)
check("closed mouths: the unit's strongest patch, average", float(np.mean([v for v, _ in res["closed"]])), 33.4, tol=4)
check("open mouths: in the top 10 somewhere", float(np.mean([t_ for _, t_ in res["open"]])), 1.0, tol=0.05)
check("closed mouths: in the top 10 somewhere", float(np.mean([t_ for _, t_ in res["closed"]])), 0.375, tol=0.1)

DRAGON = "a dragon perched on a rock, fantasy illustration"
DREG = {"upper-right": (0, 8, 8, 16), "upper-left": (0, 8, 0, 8)}
MU_OPEN = 107.03                                 # the unit's open-mouth average in the lecture's check


def paint_dragon(region, strength):
    r0, r1, c0, c1 = DREG[region]
    m = torch.zeros(16, 16, device=DEV); m[r0:r1, c0:c1] = 1
    return generate(DRAGON, 7, hooks=[(BLOCKS["down.2.1"](), paint_hook(gWd[:, 7446], m * MU_OPEN * strength))])


def dragon_share(base, im, region):
    r0, r1, c0, c1 = DREG[region]
    d = np.abs(np.asarray(im, np.float32) - np.asarray(base, np.float32)).mean(2)
    m = np.zeros((512, 512), bool); m[r0 * 32:r1 * 32, c0 * 32:c1 * 32] = True
    return float(d[m].sum() / d.sum())


dragon = generate(DRAGON, 7)
open_ur = paint_dragon("upper-right", 2.0)
s = dragon_share(dragon, open_ur, "upper-right")
print(f"upper right at 2x: {100 * s:.0f}% of the change inside a brush that is 25% of the picture")
check("the dragon, upper right at 2x: share inside", s, 0.351, tol=0.06)
fig, axs = plt.subplots(1, 2, figsize=(7, 3.6))
for a, im, ttl in zip(axs, (dragon, open_ur), ("the dragon (seed 7)", "unit 7446 painted on the upper right, 2x")):
    a.imshow(im); a.axis("off"); a.set_title(ttl, fontsize=9)
axs[1].add_patch(plt.Rectangle((256, 0), 256, 256, fill=False, ec="white", ls="--", lw=1.5))
plt.show()

### 6. Names are guesses (Plate IX)
A unit's name is someone's guess from the pictures that turn it on. Unit 3742 of block up.0.0 is called "beard". Brush it
onto the sloth's chest (the centre brush).

In [ ]:
beard = paint("up.0.0", 3742, "centre", 10)
s, _, _ = locality(sloth, beard, "centre")
print(f"'beard' on the centre: {100 * s:.0f}% of the change inside a brush that is 25% of the picture")
check("'beard' on the centre: share inside", s, 0.437, tol=0.06)
fig, ax = plt.subplots(figsize=(3.6, 3.6)); ax.imshow(beard); ax.axis("off")
ax.add_patch(plt.Rectangle((128, 128), 256, 256, fill=False, ec="white", ls="--", lw=1.5))
ax.set_title("unit 3742, 'beard', centre, strength 10", fontsize=9); plt.show()

Look at the chin. Then try *ear* (1941) and *lower jaw* (153) in the exercises: names don't always hold.

## Part 2 · Networks that see

### 7. Pictures in, labels out (Plate X)
**CLIP** (ViT-B/16) cuts a 224 × 224 photo into 14 × 14 patches of 16 × 16 pixels. Each patch becomes a list of 768
numbers, plus one *summary slot*; twelve layers mix the lists (how, is Lecture 20); the summary slot comes out. CLIP was
trained to match pictures with captions, so it labels a photo by asking which of 1,000 class sentences ("a photo of a
zebra", ...) fits best.

We fetch the course's ImageNet validation photos (the painter stays loaded: the exercises use it again).

In [ ]:
# The photos: ImageNet's 50,000 validation photos, in the exact copy the course measured (a public 256-pixel copy,
# benjamin-paine/imagenet-1k-256x256 on Hugging Face: parquet files of 25,000 photos, about 350 MB each, no key
# needed). Photo n here is photo n in the lecture. We read 100 photos at a time, only the ones we ask for.
import pyarrow.parquet as pq
from PIL import Image

SHARD = ("https://huggingface.co/datasets/benjamin-paine/imagenet-1k-256x256/resolve/main/data/"
         "validation-{:05d}-of-00002.parquet")
_files = {}


def _shard(s):
    path = fetch(SHARD.format(s), f"val-{s}.parquet", f"photo file {s}")
    if s not in _files:
        _files[s] = pq.ParquetFile(path)
    return _files[s]


def rows(ids):
    """{n: (jpeg bytes, class number 0..999)} for the photos with these numbers."""
    groups = {}
    for n in ids:
        groups.setdefault((n // 25000, (n % 25000) // 100), []).append(int(n))
    out = {}
    for (s, g), ns in sorted(groups.items()):
        t = _shard(s).read_row_group(g, columns=["image", "label"])
        ims, labs = t.column("image").to_pylist(), t.column("label").to_pylist()
        for n in ns:
            out[n] = (ims[n % 100]["bytes"], int(labs[n % 100]))
    return out


def decode(b):
    """JPEG bytes -> 256 x 256, then the centre 224 x 224, as uint8 (how Lecture 18 fed photos to CLIP and DINOv2)."""
    im = Image.open(io.BytesIO(b)).convert("RGB").resize((256, 256)).crop((16, 16, 240, 240))
    return np.asarray(im)


def labels():
    """The class of all 50,000 validation photos, in order (reads one small column)."""
    return np.concatenate([_shard(s).read(columns=["label"]).column("label").to_numpy() for s in (0, 1)])


def photos(ids):
    r = rows(ids)
    return np.stack([decode(r[n][0]) for n in ids])

In [ ]:
import open_clip
from open_clip.zero_shot_metadata import IMAGENET_CLASSNAMES, OPENAI_IMAGENET_TEMPLATES


class ViT:
    """Run a vision transformer up to the lists after layer 11 of 12 (the prism reads them), edit the patch lists,
    then finish: the last layer and the summary slot."""
    def __init__(self, kind):
        self.kind = kind
        if kind == "clip":
            with contextlib.redirect_stderr(io.StringIO()):
                m, _, _ = open_clip.create_model_and_transforms("ViT-B-16", pretrained="openai")
            self.m, self.tok = m.to(DEV).eval(), open_clip.get_tokenizer("ViT-B-16")
            self.mean, self.std, self.skip = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711), 1
        else:
            with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
                self.m = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14_reg", verbose=False).to(DEV).eval()
            self.mean, self.std, self.skip = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225), 1 + 4   # summary + 4 registers
        self.mean = torch.tensor(self.mean, device=DEV).view(1, 3, 1, 1); self.std = torch.tensor(self.std, device=DEV).view(1, 3, 1, 1)

    def prep(self, u8):
        return (torch.tensor(u8, device=DEV).permute(0, 3, 1, 2).float() / 255.0 - self.mean) / self.std

    def front(self, x):
        if self.kind == "clip":
            v = self.m.visual
            x = v.conv1(x)                                               # the patches: 768 numbers each
            x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
            cls = v.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device)
            x = v.ln_pre(torch.cat([cls, x], dim=1) + v.positional_embedding.to(x.dtype))
            bf = getattr(v.transformer, "batch_first", True)
            x = x if bf else x.permute(1, 0, 2)
            for blk in v.transformer.resblocks[:11]:
                x = blk(x)
            return x if bf else x.permute(1, 0, 2)
        x = self.m.prepare_tokens_with_masks(x)
        for blk in self.m.blocks[:11]:
            x = blk(x)
        return x

    def back(self, x):
        if self.kind == "clip":
            v = self.m.visual
            bf = getattr(v.transformer, "batch_first", True)
            y = v.transformer.resblocks[11](x if bf else x.permute(1, 0, 2))
            y = y if bf else y.permute(1, 0, 2)
            return F.normalize(v.ln_post(y[:, 0]) @ v.proj, dim=-1)
        return F.normalize(self.m.norm(self.m.blocks[11](x))[:, 0], dim=-1)


t = time.time()
clip = ViT("clip")
with torch.no_grad():
    TEXT = []
    for s in range(0, 1000, 10):                                     # 1,000 classes x 80 sentence templates, 10 classes a go
        cls = IMAGENET_CLASSNAMES[s:s + 10]
        e = F.normalize(clip.m.encode_text(clip.tok([tmp(c) for c in cls for tmp in OPENAI_IMAGENET_TEMPLATES]).to(DEV)), dim=-1)
        TEXT.append(F.normalize(e.reshape(len(cls), len(OPENAI_IMAGENET_TEMPLATES), -1).mean(1), dim=-1))
    TEXT = torch.cat(TEXT)
print(f"CLIP ViT-B/16 (OpenAI weights) and its 1,000 class sentences ready in {time.time() - t:.0f} s")

In [ ]:
LAB = labels()
first = lambda c, n=50: [int(i) for i in np.flatnonzero(LAB == c)[:n]]
ZEBRA, HORSE, TIGER = 340, 339, 292
zebras, horses = photos(first(ZEBRA)), photos(first(HORSE))
with torch.no_grad():
    lists = clip.front(clip.prep(zebras[:1]))
print("zebra photo 1, after 11 layers:", list(lists.shape), "= 1 summary slot + 14 x 14 patches, 768 numbers each")
check("lists after layer 11", list(lists.shape), [1, 197, 768])
fig, ax = plt.subplots(figsize=(3.4, 3.4)); ax.imshow(zebras[0])
for k in range(1, 14):
    ax.axhline(16 * k - 0.5, color="white", lw=0.5); ax.axvline(16 * k - 0.5, color="white", lw=0.5)
ax.axis("off"); ax.set_title("cut into 14 x 14 patches", fontsize=9); plt.show()

### 8. A prism for what CLIP sees (Plate XI)
Stevens et al. (2025) trained a prism of **24,576 units** on the lists after layer 11 (ReLU, L1 penalty). Its encoder:
`f = ReLU((x − b_dec) W_enc + b_enc)`, its decoder `x' = f W_dec + b_dec`. Which input does it expect? Try raw lists, and
lists centred and scaled (minus their average, divided by their typical length / √768), on 2,000 validation photos
(every 25th of the 50,000).

In [ ]:
def load_prism(repo):
    with open(hf_hub_download(repo, "sae.pt"), "rb") as fd:
        cfg = json.loads(fd.readline())
        sd = torch.load(io.BytesIO(fd.read()), map_location="cpu", weights_only=True)
    return {k: v.to(DEV).float() for k, v in sd.items()}


enc = lambda P, x: torch.relu((x - P["b_dec"]) @ P["W_enc"] + P["b_enc"])


def fit_sample(vit, P):
    """mean, scale, each unit's largest value and its average, and the fit on raw vs centred-scaled lists."""
    xs = []
    with torch.no_grad():
        for s in range(0, 50000, 2500):
            ids = list(range(s, s + 2500, 25))
            xs.append(vit.front(vit.prep(photos(ids)))[:, vit.skip:].reshape(-1, 768).float().cpu())
    X = torch.cat(xs)
    mu = X.mean(0); scal = float((X - mu).norm(dim=1).mean() / math.sqrt(768))
    Xs = X[torch.randperm(X.shape[0], generator=torch.Generator().manual_seed(0))[:200000]].to(DEV)
    norm = lambda x: (x - mu.to(DEV)) / scal
    out = {}
    for name, f_ in (("raw", lambda x: x), ("centred", norm)):
        se = sv = l0 = 0.0
        xm = f_(Xs).mean(0)
        with torch.no_grad():
            for i in range(0, Xs.shape[0], 8000):
                xin = f_(Xs[i:i + 8000]); f = enc(P, xin)
                se += float(((xin - (f @ P["W_dec"] + P["b_dec"])) ** 2).sum()); sv += float(((xin - xm) ** 2).sum())
                l0 += float((f > 0).float().sum())
        out[name] = (se / sv, l0 / Xs.shape[0])
    fmax = torch.zeros(P["W_enc"].shape[1], device=DEV); fsum = torch.zeros_like(fmax)
    with torch.no_grad():
        for i in range(0, Xs.shape[0], 8000):
            f = enc(P, norm(Xs[i:i + 8000])); fmax = torch.maximum(fmax, f.max(0).values); fsum += f.sum(0)
    return dict(mu=mu.to(DEV), scal=scal, norm=norm, fmax=fmax, fbg=fsum / Xs.shape[0], fit=out)


t = time.time()
P_clip = load_prism("osunlp/SAE_CLIP_24K_ViT-B-16_IN1K")
S_clip = fit_sample(clip, P_clip)
for k, (fvu, l0) in S_clip["fit"].items():
    print(f"CLIP, {k} lists: {100 * fvu:.1f}% of the variance lost, {l0:.0f} units on per patch")
print(f"({time.time() - t:.0f} s)")
check("CLIP, raw lists: variance lost", S_clip["fit"]["raw"][0], 0.2849, tol=0.01)
check("CLIP, centred and scaled: variance lost", S_clip["fit"]["centred"][0], 0.0395, tol=0.003)
check("CLIP, centred and scaled: units on per patch", S_clip["fit"]["centred"][1], 385.5, tol=8)

**The zebra unit.** For each unit, its average on the 50 zebra photos over its average on the 2,000 photos: how
*specific* it is to zebras (counting only units that are clearly on for zebras). Then where the most zebra-specific unit
fires: a box on every patch where it reaches a quarter of its largest value.

In [ ]:
def class_units(vit, P, S, ims, n=3):
    with torch.no_grad():
        fc = torch.zeros(P["W_enc"].shape[1], device=DEV); rows_ = 0
        for b in range(0, len(ims), 10):
            x = vit.front(vit.prep(ims[b:b + 10]))[:, vit.skip:].reshape(-1, 768).float()
            fc += enc(P, S["norm"](x)).sum(0); rows_ += x.shape[0]
        fc /= rows_
    spec = fc / (S["fbg"] + 1e-3)
    return torch.argsort(spec * (fc > fc.quantile(0.99)), descending=True)[:n].tolist()


zu = class_units(clip, P_clip, S_clip, zebras)
print("CLIP's three most zebra-specific units:", zu)
check("CLIP's most zebra-specific unit", zu[0], 22678)
check("CLIP's three zebra units (any order)", sorted(zu), sorted([22678, 20713, 6707]))


def unit_map(vit, P, S, ims, unit):
    with torch.no_grad():
        p = vit.front(vit.prep(ims))[:, vit.skip:]
        f = enc(P, S["norm"](p.reshape(-1, 768).float())).reshape(p.shape[0], p.shape[1], -1)[..., unit]
    g = int(round(f.shape[1] ** 0.5))
    return (f / S["fmax"][unit]).reshape(-1, g, g).cpu().numpy()


mz, mh = unit_map(clip, P_clip, S_clip, zebras[:6], zu[0]), unit_map(clip, P_clip, S_clip, horses[:6], zu[0])
fig, axs = plt.subplots(2, 6, figsize=(11, 4))
for r, (ims, mp) in enumerate(((zebras, mz), (horses, mh))):
    for c in range(6):
        axs[r, c].imshow(ims[c]); axs[r, c].axis("off")
        for (i, j) in zip(*np.nonzero(mp[c] >= 0.25)):
            axs[r, c].add_patch(plt.Rectangle((16 * j, 16 * i), 16, 16, fill=False, ec="white", lw=0.8))
fig.suptitle(f"unit {zu[0]}: boxes where it reaches a quarter of its largest value", fontsize=10); plt.show()
print(f"boxed patches: {(mz >= 0.25).sum()} on 6 zebras, {(mh >= 0.25).sum()} on 6 horses")
check("the zebra unit on horses: boxed patches", int((mh >= 0.25).sum()), 0)

### 9. Take the stripes out (Plate XII)
Steering: at every patch, encode the lists with the prism, change some units, and add the *change* back
(`x + (f_new − f) W_dec`, scaled back to the network's units), then let the last layer finish. Two ways to take the zebra
unit out: set it to **0**, or **push it against** the zebra, to −2 × its largest value (Stevens et al.'s edit). A random
unit pushed the same way is the control.

In [ ]:
def labeller_clip(emb):
    return (100.0 * emb @ TEXT.T).softmax(-1)


def run(vit, P, S, ims, edit=None, label=labeller_clip, bs=10):
    out = []
    with torch.no_grad():
        for b in range(0, len(ims), bs):
            x = vit.front(vit.prep(ims[b:b + bs]))
            if edit is not None:
                p = x[:, vit.skip:]
                f = enc(P, S["norm"](p.reshape(-1, 768).float())).reshape(p.shape[0], p.shape[1], -1)
                x = x.clone(); x[:, vit.skip:] = p + ((edit(f) - f) @ P["W_dec"] * S["scal"]).to(x.dtype)
            out.append(label(vit.back(x)))
    return torch.cat(out)


def set_to(units, value):
    s_ = torch.tensor(units, device=DEV)
    def e(f, s_=s_):
        f = f.clone(); f[..., s_] = value(s_); return f
    return e


def add(units, a, S):
    s_ = torch.tensor(units, device=DEV)
    def e(f, s_=s_):
        f = f.clone(); f[..., s_] = f[..., s_] + a * S["fmax"][s_]; return f
    return e


fm = S_clip["fmax"]
CASES = {"unedited": None, "unit set to 0": set_to(zu[:1], lambda s_: 0 * fm[s_]),
         "pushed to -2 x max": set_to(zu[:1], lambda s_: -2 * fm[s_]), "a random unit pushed": set_to([20904], lambda s_: -2 * fm[s_])}
LECTURE = {"unedited": 49, "unit set to 0": 46, "pushed to -2 x max": 0, "a random unit pushed": 49}
for name, e in CASES.items():
    p = run(clip, P_clip, S_clip, zebras, e)
    n = int((p.argmax(1) == ZEBRA).sum())
    common = IMAGENET_CLASSNAMES[int(torch.mode(p.argmax(1)).values)]
    print(f"{name:>22}: {n} of 50 called zebra (most common label: {common})")
    check(f"zebras, {name}: called zebra", n, LECTURE[name], tol=1)

### 10. Put stripes in (Plate XIII)
Now 50 sorrel horses. Add the three zebra units at *a* × their largest value, at every patch. At 1 × the label sits on a
knife edge: the lecture's two runs got 26 and 40 horses called zebra there, because each run took the units' largest values
from a different random sample of patches. Yours will land near them.

In [ ]:
LECTURE_H = {'0.0': 0, '0.5': 1, '1.0': 40, '2.0': 50}
for a in (0.0, 0.5, 1.0, 2.0):
    p = run(clip, P_clip, S_clip, horses, add(zu[:3], a, S_clip) if a else None)
    n = int((p.argmax(1) == ZEBRA).sum())
    print(f"a = {a}: {n} of 50 horses called zebra (mean P(zebra) {p[:, ZEBRA].mean():.2f})")
    if a == 1.0:
        check("horses at 1 x: called zebra (between the lecture's two runs, 26 and 40)", n, 33.0, tol=12.0)
    else:
        check(f"horses at {a} x: called zebra", n, LECTURE_H[str(a)], tol=1)

## Your turn
Each has a hint; worked solutions (with their checks) are at the end.

1. **Ear and jaw.** Paint unit 1941 ("ear") and unit 153 ("lower jaw") of block up.0.0 onto the centre at strength 10.
   Do they draw their part? *Hint: `paint("up.0.0", 1941, "centre", 10)`; look, then measure with `locality`.*
2. **Turn the dragon's head.** Paint unit 7446 on the dragon's *upper left* at 2 ×. Where does the head go, and what share
   of the change is inside? *Hint: `paint_dragon("upper-left", 2.0)`.*
3. **Three units.** Push CLIP's three zebra units to −2 × their max. How many zebras are left? *Hint: `set_to(zu, ...)`.*
4. **A tiger that becomes a cat.** Find CLIP's most tiger-specific unit on the first 50 tiger photos (class 292) and set it
   to 0. What do the tigers become? *Hint: `class_units(clip, P_clip, S_clip, photos(first(TIGER)))`.*
5. **DINOv2.** DINOv2 learned with no labels, so label a photo by its 20 nearest neighbours among 20,000 labelled photos
   (the first 20 of each class). How many zebra units must you push to remove the zebra? *Hint: build the memory with
   `ViT("dino")`; the solution shows how.*

In [ ]:
# 1. Ear and jaw
# TODO

In [ ]:
# 2. Turn the dragon's head
# TODO

In [ ]:
# 3. Three units
# TODO

In [ ]:
# 4. A tiger that becomes a cat
# TODO

In [ ]:
# 5. DINOv2
# TODO

## Solutions

In [ ]:
# 1. Neither draws its part: they paint fur and cloth over the shirt.
fig, axs = plt.subplots(1, 3, figsize=(10, 3.4))
axs[0].imshow(sloth); axs[0].set_title("before", fontsize=9)
for a, (unit, name) in zip(axs[1:], ((1941, "ear"), (153, "lower jaw"))):
    im = paint("up.0.0", unit, "centre", 10)
    s, _, _ = locality(sloth, im, "centre")
    a.imshow(im); a.set_title(f"unit {unit}, '{name}': {100 * s:.0f}% inside", fontsize=9)
    print(f"'{name}': {100 * s:.0f}% of the change inside the centre brush")
for a in axs:
    a.axis("off")
plt.show()
check("'ear' on the centre: share inside", locality(sloth, paint("up.0.0", 1941, "centre", 10), "centre")[0], 0.433, tol=0.06)

In [ ]:
# 2. The head turns into the painted corner, mouth open: the unit also decides where the head goes.
ul = paint_dragon("upper-left", 2.0)
s = dragon_share(dragon, ul, "upper-left")
fig, ax = plt.subplots(figsize=(3.6, 3.6)); ax.imshow(ul); ax.axis("off")
ax.add_patch(plt.Rectangle((0, 0), 256, 256, fill=False, ec="white", ls="--", lw=1.5)); ax.set_title(f"upper left, 2x: {100 * s:.0f}% inside", fontsize=9)
plt.show()
check("the dragon, upper left at 2x: share inside", s, 0.311, tol=0.06)
del pipe, unet, prisms, sd, gWe, gWd                 # done painting: free the GPU for DINOv2
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 3. All three pushed: no zebras at all.
p = run(clip, P_clip, S_clip, zebras, set_to(zu, lambda s_: -2 * fm[s_]))
n = int((p.argmax(1) == ZEBRA).sum())
print(f"three zebra units pushed: {n} of 50 called zebra")
check("zebras, three units pushed: called zebra", n, 0, tol=1)

In [ ]:
# 4. Zeroing the tiger unit: tigers become "tiger cat", a sensible near miss.
tigers = photos(first(TIGER))
tu = class_units(clip, P_clip, S_clip, tigers, n=1)
print("CLIP's most tiger-specific unit:", tu)
check("CLIP's most tiger-specific unit", tu[0], 15500)
p0, p1 = run(clip, P_clip, S_clip, tigers), run(clip, P_clip, S_clip, tigers, set_to(tu, lambda s_: 0 * fm[s_]))
n0, n1 = int((p0.argmax(1) == TIGER).sum()), int((p1.argmax(1) == TIGER).sum())
now = IMAGENET_CLASSNAMES[int(torch.mode(p1.argmax(1)).values)]
print(f"tigers called tiger: {n0} of 50 unedited, {n1} with the unit set to 0; now mostly '{now}'")
check("tigers, unedited", n0, 43, tol=1)
check("tigers, tiger unit set to 0", n1, 2, tol=1)
check("what they become", now, 'tiger cat')

In [ ]:
# 5. DINOv2: its zebra is spread over more units. One pushed is not enough; three are.
t = time.time()
del clip, TEXT; gc.collect(); torch.cuda.empty_cache()
dino = ViT("dino")
P_dino = load_prism("osunlp/SAE_DINOv2_24K_ViT-B-14_IN1K")
S_dino = fit_sample(dino, P_dino)
print(f"DINOv2, centred and scaled: {100 * S_dino['fit']['centred'][0]:.1f}% lost, {S_dino['fit']['centred'][1]:.0f} units on")
check("DINOv2, centred and scaled: variance lost", S_dino["fit"]["centred"][0], 0.069, tol=0.004)
from concurrent.futures import ThreadPoolExecutor
mem_ids = sorted(int(i) for c in range(1000) for i in np.flatnonzero(LAB == c)[:20])
MEM, MEML = [], []
pool = ThreadPoolExecutor(4)
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):   # half precision: the memory only needs directions
    for b in range(0, len(mem_ids), 500):
        ids = mem_ids[b:b + 500]
        r = rows(ids)
        ims = np.stack(list(pool.map(lambda n: decode(r[n][0]), ids)))
        for k in range(0, len(ids), 250):
            MEM.append(dino.back(dino.front(dino.prep(ims[k:k + 250]))).float())
        MEML += [int(LAB[i]) for i in ids]
MEM = F.normalize(torch.cat(MEM), dim=-1); MEML = torch.tensor(MEML, device=DEV)


def labeller_dino(emb, k=20):
    s_ = emb @ MEM.T
    v, i = s_.topk(k + 1, dim=1)
    v, i = v[:, 1:], i[:, 1:]                        # drop the nearest, which is the photo itself when it is in memory
    p = torch.zeros(emb.shape[0], 1000, device=DEV)
    p.scatter_add_(1, MEML[i], torch.exp(v / 0.07))
    return p / p.sum(1, keepdim=True)


print(f"DINOv2's memory of {len(MEM):,} labelled photos ready ({time.time() - t:.0f} s)")
du = class_units(dino, P_dino, S_dino, zebras)
check("DINOv2's most zebra-specific unit", du[0], 1283)
fd = S_dino["fmax"]
for name, e, exp in (("unedited", None, 49), ("one unit pushed", set_to(du[:1], lambda s_: -2 * fd[s_]), 46),
                     ("three units pushed", set_to(du, lambda s_: -2 * fd[s_]), 1)):
    p = run(dino, P_dino, S_dino, zebras, e, label=labeller_dino)
    n = int((p.argmax(1) == ZEBRA).sum())
    print(f"DINOv2, {name}: {n} of 50 called zebra")
    check(f"DINOv2 zebras, {name}: called zebra", n, exp, tol=1)
print(f"whole notebook: {(time.time() - T0) / 60:.1f} minutes")